In [29]:
%pip install -r "requirements.txt"

from did_functions import *
import pandas as pd
import kagglehub
import geopandas as gpd
import datetime as dt
import sqlite3
import os

Note: you may need to restart the kernel to use updated packages.


### Hardcoded Values
- NCIC Codes are taken from data since they overlap with Vision Zero data for the period of interest

In [30]:
## Hardcoded Values depending on analysis

# Vision Zero city codes
codes_to_drop = ['1942', '3711', '4313', '0105']

# data paths
crash_path = "alexgude/california-traffic-collision-data-from-switrs"
ncic_path = r"../data/raw_data/NCIC Code Jurisdiction List_04242023 - Sheet1.csv"
final_data_path = r"../data/clean_data/California_Collisions_Clean.csv"

date_col = "collision_date"

Load data from the online source

In [31]:
# Download latest data
data = kagglehub.dataset_download(crash_path)
db_file_path = os.path.join(data, 'switrs.sqlite')
ncic = load_data(ncic_path)

NCIC Code Jurisdiction List_04242023 - Sheet1.csv converted to dataframe.


## Data Cleaning

### Grab data for synthetic control

In [32]:
# establish sql connection to database path
conn = sqlite3.connect(db_file_path)
# use sql query to pull data
collisions = pd.read_sql_query("""
    SELECT case_id, collision_date, county_city_location, collision_severity, county_location, population, weather_1, primary_collision_factor, pcf_violation_category, hit_and_run, type_of_collision, road_condition_1, road_surface, lighting, alcohol_involved, latitude, longitude
    FROM collisions
    WHERE collision_date >= '2010-01-01' AND collision_date <= '2016-12-31';
""", conn)

### Organize Cities by NCIC Codes

In [33]:
# combine data sets
collisions_coded = pd.merge(collisions, ncic, how='left', left_on='county_city_location', right_on='Code')
# drop vision zero codes from synthetic data
collisions_ca = collisions_coded[~collisions_coded['Code'].isin(codes_to_drop)]

### Label treatment by city and time


In [34]:
collisions_ca[date_col] = pd.to_datetime(collisions_ca[date_col])
collisions_ca['year_month'] = collisions_ca[date_col].dt.to_period('M')

Remove unnecessary entries from agency column

In [35]:
# Get rid of Sheriff's Departments in agencies
collisions_ca = collisions_ca[~collisions_ca['Agency'].str.contains('Sheriff|Police', na=False)]

In [36]:
# Find top 18 cities plus San Francisco in terms of crashes by Agency Name
top_cities = collisions_ca.groupby('Agency')['case_id'].count().sort_values(ascending = False)
top_cities = top_cities.head(19)

In [37]:
# save final dataset
collisions_ca.to_csv(final_data_path)